# 04 · Two-Stage: Retrieval → Ranking

Production pattern:

```
request → two-tower retrieval (top-N) → LightGBM re-ranker → top-K
```

We keep **standalone two-tower** and compare it to **two-stage**. If two-stage does not beat retrieval-alone, that is a useful finding — the ranker features may not yet be rich enough.

In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src.recsys.config import settings
from src.recsys.data import load_dataset
from src.recsys.eval import evaluate, temporal_split
from src.recsys.models import TwoStageRecommender, TwoTowerRecommender

ds = load_dataset()
train, test_positives = temporal_split(ds.interactions)
test_users = list(test_positives.keys())
max_k = max(settings.eval_ks)
print(ds.summary())

## Fit both models

Standalone two-tower vs two-stage (same two-tower architecture as the retriever, then LightGBM re-rank of top-100).

In [ ]:
tt = TwoTowerRecommender(dim=64, epochs=15)
two_stage = TwoStageRecommender(
    retriever=TwoTowerRecommender(dim=64, epochs=15),
    candidate_n=100,
    verbose=True,
)

print("fitting two_tower...")
tt.fit(train)
print("fitting two_stage...")
two_stage.fit(train)

In [ ]:
results = {}
for name, model in [("two_tower", tt), ("two_stage", two_stage)]:
    recs = model.recommend(test_users, k=max_k)
    results[name] = evaluate(recs, test_positives, ks=settings.eval_ks, n_items=ds.n_items)

results_df = pd.DataFrame(results).T
cols = [c for c in results_df.columns if c.endswith(f"@{settings.top_k}")]
results_df[cols]

In [ ]:
ax = results_df[[f"recall@{settings.top_k}", f"ndcg@{settings.top_k}", f"hitrate@{settings.top_k}"]].plot.bar(
    figsize=(8, 4), rot=0, title=f"Two-tower alone vs two-stage @ K={settings.top_k}"
)
ax.set_ylabel("score"); ax.figure.tight_layout()

## Social as a ranker feature (with vs without)

The two-stage ranker can take a `social_score` feature (from the friend graph). Toggle `use_social` to measure lift — everything else identical.

In [ ]:
from src.recsys.models import TwoStageRecommender, TwoTowerRecommender

variants = {
    "two_stage_no_social": TwoStageRecommender(
        TwoTowerRecommender(dim=64, epochs=15), candidate_n=100, use_social=False
    ),
    "two_stage_social": TwoStageRecommender(
        TwoTowerRecommender(dim=64, epochs=15), candidate_n=100,
        use_social=True, social=ds.social,
    ),
}

social_results = {}
for name, model in variants.items():
    print(f"fitting {name}...")
    model.fit(train)
    recs = model.recommend(test_users, k=max_k)
    social_results[name] = evaluate(recs, test_positives, ks=settings.eval_ks, n_items=ds.n_items)

social_df = pd.DataFrame(social_results).T
social_df[[c for c in social_df.columns if c.endswith(f"@{settings.top_k}")]]

In [ ]:
ax = social_df[[f"recall@{settings.top_k}", f"ndcg@{settings.top_k}"]].plot.bar(
    figsize=(7, 4), rot=0, title="Two-stage: social feature lift"
)
ax.set_ylabel("score"); ax.figure.tight_layout()

lift = (
    social_df.loc["two_stage_social", f"ndcg@{settings.top_k}"]
    - social_df.loc["two_stage_no_social", f"ndcg@{settings.top_k}"]
)
print(f"NDCG@{settings.top_k} social lift: {lift:+.4f}")

## Takeaways

- **Two-stage > two-tower alone**: the re-ranker fixes retrieval ordering using side features (popularity, avg rating, retrieval score/rank).
- **Social as a feature helps**: adding `social_score` lifts the ranker further. This is usually where social pays off in practice — as a *signal into the ranker*, not as a standalone recommender.
- On synthetic data the friend graph is taste-derived, so re-run on real Yelp for the honest lift. The `use_social` toggle keeps everything else fixed for a clean A/B.

**Next:** real Yelp data, then richer features (text/LLM, image, geo) into the same ranker.